# Reranker experiments — completing the paper's evidence — corrected

Four outstanding items sharing one expensive setup (corpus embedding + retrieval), so the index is
built **once**.

| | Item | Why |
|---|---|---|
| **A** | Fix `bge-reranker-base` | It scored *below* the no-rerank baseline — the signature of a 2D-score-array bug |
| **B** | Arabic-specific rerankers | `GATE-Reranker-V1`, `Namaa-ARA-Reranker-V1` |
| **C** | Rerank depth ablation | top-10/20/50 |
| **D** | **Asymmetry test** | paired difference-of-differences bootstrap for "reranking helps Darija more" |

## Defects fixed

**Paths.** The original opened `corpus_v2.json` / `qa_pairs_wiki.json` from the working directory;
in this repo they are `data/corpus.json` and `data/qa_pairs_wiki.json`. Outputs went to the working
directory rather than `results/`, and the last cell was a bare `from google.colab import files`,
which raises outside Colab and aborts the notebook at the very end. All resolved: repo → working
directory → GitHub, with the source printed, and the download guarded.

**Deprecated `torch_dtype`.** `automodel_args={"torch_dtype": ...}` is deprecated in current
transformers and absent in old ones. Replaced with a post-load cast that works on every version.

**`primary_depth` was not guaranteed to be among `depths`.** Sections A, B and D all index
`results[(method, field, primary_depth)]`. Set `primary_depth` to a value not in `depths` and
section D's `except KeyError: continue` skips *every* method and prints an empty asymmetry table —
which reads as "no effect" rather than as a configuration error. `primary_depth` is now asserted
into `depths` up front, and D reports any method it had to skip instead of swallowing it.

**The 2D-score guard was already correct** and is kept as-is — that was the point of item A.

### Install

In [ ]:
import importlib.util, subprocess, sys

need = [p for p, m in [("rank_bm25", "rank_bm25"), ("sentence-transformers", "sentence_transformers"), ("transformers", "transformers")] if importlib.util.find_spec(m) is None]
if need:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], check=False)

print("deps ready")

### Paths

In [ ]:
import os, json, re, gc, time, random, pickle, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd

RAW_BASE = "https://raw.githubusercontent.com/Rania-khaoudane/MSA/main/data"

def _repo_root():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "data" / "qa_pairs_wiki.json").exists():
            return base
    return None

ROOT = _repo_root()

def find_file(*names, subdirs=("data", "results")):
    """Locate a file: repo subdirs first, then the working directory."""
    for n in names:
        if ROOT:
            for sd in subdirs:
                p = ROOT / sd / n
                if p.exists():
                    return p
        if Path(n).exists():
            return Path(n)
    return None

def resolve(*names):
    """Local -> GitHub. Returns (json, description)."""
    p = find_file(*names, subdirs=("data",))
    if p:
        return json.loads(p.read_text(encoding="utf-8")), f"local: {p}"
    for n in names:
        try:
            url = f"{RAW_BASE}/{n}"
            with urllib.request.urlopen(url) as r:
                return json.loads(r.read().decode("utf-8")), f"github: {url}"
        except Exception:
            continue
    raise FileNotFoundError(f"none of {names} found locally or on GitHub")

OUT_DIR = (ROOT / "results") if ROOT else Path(".")
OUT_DIR.mkdir(exist_ok=True)
def out(name):
    return str(OUT_DIR / name)

print("repo root :", ROOT or "(not in the repo)")
print("output dir:", OUT_DIR.resolve())

import torch

### Config

In [ ]:
CONFIG = {
    "base_encoder": "intfloat/multilingual-e5-base",
    "alpha": 0.8,
    "depths": [10, 20, 50],          # C: rerank depth ablation
    "primary_depth": 20,             # depth used for A, B and D
    "rerankers": [
        "BAAI/bge-reranker-v2-m3",              # current best, for comparison
        "BAAI/bge-reranker-base",               # A: rerun with the scoring guard
        "NAMAA-Space/GATE-Reranker-V1",         # B: Arabic-specific, claims dialect coverage
        "NAMAA-Space/Namaa-ARA-Reranker-V1",    # B: second Arabic-specific option
    ],
    "k_values": (1, 3, 5, 10),
    "bootstrap_n": 1000,
    "seed": 42,
    "max_length": 512,
    "batch_size": 16,
    "checkpoint": "reranker_experiments_checkpoint.pkl",
}

# A, B and D all index results[(method, field, primary_depth)]. If primary_depth
# were not among depths, D's `except KeyError: continue` would silently skip every
# method and print an empty asymmetry table that reads as "no effect".
assert CONFIG["primary_depth"] in CONFIG["depths"], \
    f"primary_depth {CONFIG['primary_depth']} must be one of depths {CONFIG['depths']}"
CONFIG

### Load data

In [ ]:
corpus, src_c = resolve("corpus_v2.json", "corpus.json")
wiki_qa, src_q = resolve("qa_pairs_wiki.json")
print("corpus from", src_c); print("qa     from", src_q)

corpus_ids = [c["chunk_id"] for c in corpus]
corpus_texts = [c["text"] for c in corpus]
corpus_map = dict(zip(corpus_ids, corpus_texts))
known = set(corpus_ids)
qa = [q for q in wiki_qa if q["source_chunk_id"] in known]
print(f"Corpus {len(corpus)} | evaluating all {len(qa)} items")

### BM25

In [ ]:
from rank_bm25 import BM25Okapi

DIAC = re.compile(r"[\u0610-\u061A\u064B-\u065F\u06D6-\u06DC\u06DF-\u06E8\u06EA-\u06ED\u0670]")

def normalize_arabic(t):
    t = DIAC.sub("", t)
    t = re.sub(r"[\u0625\u0623\u0622\u0627]", "\u0627", t)
    t = re.sub(r"\u0649", "\u064A", t); t = re.sub(r"\u0629", "\u0647", t)
    t = re.sub(r"\u0624", "\u0648", t); t = re.sub(r"\u0626", "\u064A", t)
    t = re.sub(r"\u0640+", "", t); t = re.sub(r"[^\w\s]", " ", t)
    return re.sub(r"\s+", " ", t).strip()

def tokenize(t):
    return normalize_arabic(t).split()

bm25 = BM25Okapi([tokenize(t) for t in corpus_texts])
_bm = {}
def bm25_scores(q):
    if q not in _bm:
        _bm[q] = np.asarray(bm25.get_scores(tokenize(q)))
    return _bm[q]

def minmax(a):
    lo, hi = a.min(), a.max()
    return (a - lo) / (hi - lo) if hi > lo else np.zeros_like(a)

print("BM25 index built")

### Retrieve once at the DEEPEST depth; shallower depths are prefixes

In [ ]:
from sentence_transformers import SentenceTransformer

MAXD = max(CONFIG["depths"] + [CONFIG["primary_depth"]])
print(f"Building index and retrieving top-{MAXD} (shallower depths are prefixes)...")

bi = SentenceTransformer(CONFIG["base_encoder"])
corpus_emb = np.asarray(
    bi.encode([f"passage: {t}" for t in corpus_texts],
              normalize_embeddings=True, batch_size=32, show_progress_bar=True), "float32")

def retrieve(query, k):
    q = bi.encode([f"query: {query}"], normalize_embeddings=True)[0]
    s = (CONFIG["alpha"] * minmax(corpus_emb @ q)
         + (1 - CONFIG["alpha"]) * minmax(bm25_scores(query)))
    return [corpus_ids[i] for i in np.argsort(-s)[:k]]

candidates = {}
for field in ["msa_query", "darija_query"]:
    candidates[field] = {q["id"]: retrieve(q[field], MAXD) for q in qa}
    print(f"  {field} done")

del bi, corpus_emb
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

### Evaluation helper + baselines

In [ ]:
def evaluate_order(ordered_by_qid):
    o = {f"R@{k}": [] for k in CONFIG["k_values"]}
    rr = []
    for q in qa:
        ordered = ordered_by_qid[q["id"]]
        gold = q["source_chunk_id"]
        pos = ordered.index(gold) + 1 if gold in ordered else None
        for k in CONFIG["k_values"]:
            o[f"R@{k}"].append(1.0 if (pos and pos <= k) else 0.0)
        rr.append(1.0 / pos if pos else 0.0)
    return {**{k: np.array(v) for k, v in o.items()}, "MRR": np.array(rr)}

CKPT = out(CONFIG["checkpoint"])
results = {}
if os.path.exists(CKPT):
    with open(CKPT, "rb") as f:
        results.update(pickle.load(f))
    print(f"Resumed {len(results)} cached entries.")

for depth in CONFIG["depths"]:
    for field in ["msa_query", "darija_query"]:
        key = ("no_rerank", field, depth)
        if key not in results:
            results[key] = evaluate_order({q["id"]: candidates[field][q["id"]][:depth] for q in qa})

print("\nBaseline (no reranking):")
for depth in CONFIG["depths"]:
    m = results[("no_rerank", "darija_query", depth)]
    print(f"  top-{depth:<3} Darija R@1={m['R@1'].mean():.3f}  R@5={m['R@5'].mean():.3f}")

### A + B + C: every reranker at every depth, with the scoring guard

In [ ]:
from sentence_transformers import CrossEncoder

def score_pairs_ce(ce, query, cand_ids):
    """THE BUG FIX (item A): some cross-encoders return a 2D per-class array
    instead of a 1D relevance score. argsort on 2D sorts within rows, producing
    garbage ordering -- which is what put bge-reranker-base below baseline."""
    pairs = [(query, corpus_map[c]) for c in cand_ids]
    scores = np.asarray(ce.predict(pairs, batch_size=CONFIG["batch_size"], show_progress_bar=False))
    if scores.ndim > 1:
        scores = scores[:, -1]
    return scores

def run_reranker(model_name):
    short = model_name.split("/")[-1]
    if all((short, f, d) in results for d in CONFIG["depths"] for f in ["msa_query", "darija_query"]):
        print(f"\n=== {short} === (cached, skipping)"); return
    print(f"\n=== {short} ===")
    try:
        ce = CrossEncoder(model_name, max_length=CONFIG["max_length"], trust_remote_code=True)
        try:
            ce.model = ce.model.to(dtype=torch.float32)
        except Exception:
            pass
    except Exception as e:
        print(f"  SKIPPED (load): {type(e).__name__}: {str(e)[:160]}"); return
    try:
        for field in ["msa_query", "darija_query"]:
            full = {q["id"]: (candidates[field][q["id"]],
                              score_pairs_ce(ce, q[field], candidates[field][q["id"]])) for q in qa}
            for depth in CONFIG["depths"]:
                reordered = {}
                for qid, (cands, scores) in full.items():
                    sub_c, sub_s = cands[:depth], scores[:depth]
                    reordered[qid] = [sub_c[i] for i in np.argsort(-sub_s)]
                results[(short, field, depth)] = evaluate_order(reordered)
            print(f"  {field} done (all depths)")
    except Exception as e:
        print(f"  SKIPPED (inference): {type(e).__name__}: {str(e)[:160]}")
    finally:
        del ce
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    with open(CKPT, "wb") as f:
        pickle.dump(results, f)

for name in CONFIG["rerankers"]:
    run_reranker(name)

### Results table

In [ ]:
rows = [{"method": m, "query": f, "depth": d, **{k: float(v.mean()) for k, v in mm.items()}}
        for (m, f, d), mm in results.items()]
df = pd.DataFrame(rows)
df.to_csv(out("reranker_experiments_full.csv"), index=False)

D = CONFIG["primary_depth"]
print("=" * 90); print(f"RESULTS AT TOP-{D}"); print("=" * 90)
print(df[df.depth == D].pivot(index="method", columns="query", values=["R@1", "R@5", "MRR"])
      .to_string(float_format=lambda x: f"{x:.3f}"))

### A: did the bug fix change bge-reranker-base?

In [ ]:
print("\n" + "=" * 90); print("A. BUG FIX CHECK - bge-reranker-base"); print("=" * 90)
prev = {"msa_query": 0.520, "darija_query": 0.365}   # from the buggy run
sub = df[(df.method == "bge-reranker-base") & (df.depth == D)]
if len(sub):
    for _, r in sub.iterrows():
        base = df[(df.method == "no_rerank") & (df["query"] == r["query"]) & (df.depth == D)]["R@1"].iloc[0]
        print(f"  {r['query']:<14} before fix {prev[r['query']]:.3f} | after fix {r['R@1']:.3f} | "
              f"no-rerank baseline {base:.3f}")
    print("\n  At or above baseline -> the 2D-score bug explained it.")
    print("  Still below -> the model is genuinely unsuited; report it rather than dropping it.")
else:
    print("  Model did not run.")

### C: depth ablation

In [ ]:
print("\n" + "=" * 90); print("C. RERANK DEPTH ABLATION (Darija R@1)"); print("=" * 90)
print(df[df["query"] == "darija_query"].pivot(index="depth", columns="method", values="R@1")
      .to_string(float_format=lambda x: f"{x:.3f}"))
print("""
Deeper reranking raises the ceiling but costs proportionally more cross-encoder
passes. Flat or falling numbers with depth mean the extra candidates add noise.""")

### D: the asymmetry test (the paper's novel claim)

In [ ]:
rng = np.random.default_rng(CONFIG["seed"])

def boot_ci(d):
    idx = rng.integers(0, len(d), size=(CONFIG["bootstrap_n"], len(d)))
    m = d[idx].mean(axis=1)
    return d.mean(), *np.percentile(m, [2.5, 97.5])

print("\n" + "=" * 90)
print("D. ASYMMETRY TEST - does reranking help Darija MORE than MSA?")
print("=" * 90)
print("Two separate CIs on the two gains do NOT establish that the gains differ.")
print("This is a paired difference-of-differences bootstrap, which does.\n")

asym, skipped = [], []
for method in df.method.unique():
    if method == "no_rerank":
        continue
    try:
        dar = results[(method, "darija_query", D)]["R@1"] - results[("no_rerank", "darija_query", D)]["R@1"]
        msa = results[(method, "msa_query", D)]["R@1"] - results[("no_rerank", "msa_query", D)]["R@1"]
    except KeyError:
        skipped.append(method)     # reported below, not silently swallowed
        continue
    dd, lo, hi = boot_ci(dar - msa)
    asym.append({"method": method, "darija_gain": dar.mean(), "msa_gain": msa.mean(),
                 "difference": dd, "lo": lo, "hi": hi,
                 "verdict": ("helps Darija more" if lo > 0 else
                             "helps MSA more" if hi < 0 else "no significant asymmetry")})

if skipped:
    print(f"  NOTE: no results at top-{D} for {', '.join(skipped)} - excluded from this test.\n")
asymdf = pd.DataFrame(asym)
print(asymdf.to_string(index=False, float_format=lambda x: f"{x:+.3f}") if len(asymdf)
      else "  No method had results at this depth - nothing to test.")
asymdf.to_csv(out("asymmetry_test.csv"), index=False)
print("""
'helps Darija more' with a CI excluding zero is what the paper's novel claim
requires. 'no significant asymmetry' means the claim must be softened to a
descriptive observation rather than a demonstrated effect.""")

### Dialect gap by method

In [ ]:
print("\n" + "=" * 90); print(f"DIALECT GAP AT TOP-{D} (MSA - Darija, R@1)"); print("=" * 90)
gaps = []
for method in df.method.unique():
    try:
        d = results[(method, "msa_query", D)]["R@1"] - results[(method, "darija_query", D)]["R@1"]
    except KeyError:
        continue
    g, lo, hi = boot_ci(d)
    gaps.append({"method": method, "gap": g, "lo": lo, "hi": hi, "significant": "yes" if lo > 0 else "no"})
gapdf = pd.DataFrame(gaps).sort_values("gap")
print(gapdf.to_string(index=False, float_format=lambda x: f"{x:+.3f}"))
gapdf.to_csv(out("gap_by_reranker.csv"), index=False)

print(f"\nAll outputs written under: {OUT_DIR.resolve()}")
for f in ["reranker_experiments_full.csv", "asymmetry_test.csv", "gap_by_reranker.csv"]:
    print("  ", f)

# The original ended with a bare `from google.colab import files`, which raises
# outside Colab and aborted the notebook on the final cell.
try:
    from google.colab import files as colab_files
    for f in ["reranker_experiments_full.csv", "asymmetry_test.csv", "gap_by_reranker.csv"]:
        colab_files.download(out(f))
except ImportError:
    print("(Not in Colab - files are on disk at the path above.)")